# 05 · Application web & visualisations (Personne 5)

**Objectif.** Ce notebook documente la partie « interface » du projet :
1. le **questionnaire** Streamlit (17 questions, les 5 types exigés par le sujet) et le fichier `answers.json` qu'il produit ;
2. le branchement sur le moteur de recommandation (`src/recommender.recommend`, P4) ;
3. les **5 visualisations** de `src/viz.py` : ce que chacune montre et pourquoi elle rend le résultat **interprétable** ;
4. les captures d'écran de l'application (section 6).

**Place dans le pipeline (PLAN §3).**
```
Questionnaire (app/streamlit_app.py) ──► answers.json ──► recommend() (P4) ──► UserProfile + top 3 Recommendation
                                                                                     │
                                                         src/viz.py ◄────────────────┘  (radar, barres, carte, heatmap)
```

**Lancer l'application** (depuis la racine du repo) :
```bash
python -m scripts.prepare_app      # une fois : encode les livres + calcule la carte 2D (évite l'attente pendant la démo)
streamlit run app/streamlit_app.py
```

In [ ]:
# Étape 0 : imports. Le notebook est dans notebooks/ ; on ajoute la racine du repo au chemin
# pour importer src/ (P1-P4) et app/ (P5).
import json
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.io as pio

from app import questions as Q
from src import config, viz
from src.data_loader import load_books
from src.embeddings import EmbeddingEngine

pio.renderers.default = "notebook_connected"   # figures interactives dans Jupyter / VS Code
print("Racine du repo :", ROOT)

## 1. Le questionnaire : 17 questions, 5 types de questions

Le sujet exige des questions **ouvertes, Likert, guidées, à choix multiple et à cases à cocher**, avec « suffisamment de texte libre ».
Le contenu des questions est défini dans `app/questions.py` (sans Streamlit, donc testable et réutilisable par P4/P6) ; `app/streamlit_app.py` ne fait que l'affichage.

| Étape de l'app | Questions | Type exigé | Widget Streamlit | Champ de `answers.json` |
|---|---|---|---|---|
| 1. Tell us about your reading | Q1 (obligatoire, ≥ 30 car.), Q2, Q3, Q4 | **Texte libre** | `st.text_area` | `free_text.*` |
| 2. Genres & themes | Q5 (21 genres), Q6 (20 thèmes) | **Cases à cocher** | `st.checkbox` (+ bulle d'aide = phrase de `blocks.json`) | `genres`, `themes` |
| 3. Context | Q7 époque du récit | **Choix multiple** (radio) | `st.radio` | `period` |
| | Q8 public | **Choix multiple** (liste) | `st.selectbox` | `audience` |
| | Q9 formats | Choix multiple | `st.multiselect` | `formats` |
| 4. Your taste | Q10-Q13 complexité, rythme, tonalité, popularité ; Q14 importance des 4 blocs | **Likert 1-5** | `st.slider` + libellés aux extrémités | `likert.*` |
| 5. Guided questions | Q15 auteur favori → champ conditionnel ; Q16 personnages/intrigue ; Q17 exclure l'auteur | **Guidées** | `st.radio` + `st.text_input` conditionnel + `st.checkbox` | `favorite_author`, `guided.*` |

**Choix de conception :**
- **Une seule source de vérité pour les labels** : les cases Q5/Q6 et les époques Q7 sont lues dans `data/reference/blocks.json` (P2). Si P2 modifie un label, l'interface suit automatiquement, et P4 retrouve directement la phrase descriptive à encoder.
- **Q7 renvoie un label du bloc Atmosphere** (« 20th century », « Future »…) : c'est l'époque du **récit**, traitée sémantiquement. Ce n'est pas un filtre sur `pub_year` (année d'édition), qui n'a rien à voir (cf. avertissement de P1 dans `config.py`).
- **Auteur favori vérifié en direct** avec `find_author` de P1 (matching flou) : l'utilisateur voit tout de suite si l'auteur est dans le catalogue.
- **Navigation multi-étapes** avec `st.session_state`, barre de progression, validation à chaque étape (Q1 obligatoire), boutons Précédent / Suivant.
- **Sauvegarde** : chaque envoi est écrit dans `data/responses/answers_<horodatage>.json` (consigne « store responses in a structured format ») et téléchargeable.

In [ ]:
# Étape 1 : construire answers.json pour un persona de démo, exactement comme le fait l'app
blocks = Q.load_blocks()
state = Q.persona_state("Dark academia reader", blocks)   # ce que l'utilisateur a saisi
print("Erreurs de validation :", Q.validate_all(state))     # [] = formulaire valide
answers = Q.build_answers(state, blocks)
print(json.dumps(answers, indent=2, ensure_ascii=False))

In [ ]:
# Étape 2 : la validation bloque un Q1 trop court (le texte libre est le cœur de l'analyse sémantique)
empty = Q.default_state(blocks)
empty["q1_ideal"] = "a good book"
Q.validate_step(0, empty)

## 2. Appel du moteur de recommandation

L'app appelle `src.recommender.recommend(answers, top_k=3)` (contrat PLAN §4.4). Tant que P4 n'a pas livré ce module,
elle utilise `app/stub_recommender.py`, qui renvoie des objets **au même format** (`UserProfile`, `Recommendation`),
calculés avec une formule simplifiée sur les vraies données (embeddings de P3, `label_scores` de P2, filtres de P1).
Le basculement est automatique : il suffit que `src/recommender.py` existe.

> ⚠️ Si la cellule suivante affiche `stub`, les scores ci-dessous ne sont **pas** ceux du projet final : relancer le notebook une fois P4 intégré.

In [ ]:
# Étape 3 : moteur final (P4) s'il est intégré, sinon EmbeddingEngine (P3) + stub.
# Premier lancement : téléchargement du modèle + encodage des 12 000 livres (mis en cache ensuite).
try:
    from src.recommender import get_recommender, recommend   # moteur final (P4)
    MODE = "P4"
    reco = get_recommender()
    engine, books = reco.engine, reco.books
    profile, recs = recommend(answers, top_k=3)
except ImportError:
    from app.stub_recommender import recommend                # remplaçant provisoire (même contrat)
    MODE = "stub"
    engine = EmbeddingEngine(books=load_books())
    books = engine.books
    profile, recs = recommend(answers, top_k=3, engine=engine, books=books)
matrix = engine.book_matrix()

print("Mode :", MODE)
print("Profil :", profile.summary)
pd.DataFrame([{"rank": r.rank, "title": r.title, "authors": r.authors, "final_score": round(r.final_score, 3),
               "match": r.match_label, "coverage": round(r.coverage, 3)} for r in recs])

## 3. Visualisations (`src/viz.py`)

Toutes les fonctions de `viz.py` sont **pures** : elles prennent des données déjà calculées et renvoient une figure Plotly.
La même fonction sert dans l'app, dans ce notebook et pour les figures du rapport (export PNG).
**Charte commune** : une couleur fixe par bloc dans toutes les figures (Genre bleu, Thèmes vert, Style orange, Atmosphère violet, résumé gris, pénalité rouge), pour que le lecteur fasse le lien d'un graphique à l'autre et avec les puces de labels des cartes de recommandation.

### 3.1 Radar du profil lecteur — `radar_profile(profile.block_affinities)`
**Ce qu'il montre :** pour chacun des 4 blocs, les labels les plus proches des réponses de l'utilisateur (cosinus entre son texte et la phrase descriptive du label).
**Pourquoi c'est interprétable :** l'utilisateur vérifie d'un coup d'œil que le système a « compris » sa demande (ex. une demande sur une vieille université fait ressortir *Academic & campus* sans que le mot ait été écrit). Un radar par bloc plutôt qu'un seul : ~66 labels sur un même radar seraient illisibles, et la séparation reflète la structure du référentiel.

In [ ]:
fig_profile = viz.radar_profile(profile.block_affinities)
fig_profile.show()

### 3.2 Radar « vous vs les livres » — `radar_compare(profile.weights, {titre: block_scores})`
**Ce qu'il montre :** en pointillés, l'importance que l'utilisateur donne à chaque bloc (Likert Q14, ramené sur [0, 1]) ; en couleur, le score calibré de chaque livre sur ce bloc (et sur la similarité au résumé).
**Lecture :** quand le polygone du livre couvre celui de l'utilisateur, le livre satisfait les critères jugés importants ; un creux sur un axe important explique un score plus faible. C'est la traduction visuelle du **score de couverture** Σ W_b·S_b / Σ W_b.

In [ ]:
fig_compare = viz.radar_compare(profile.weights, {r.title: r.block_scores for r in recs})
fig_compare.show()

### 3.3 Décomposition du score — `score_breakdown(recs)`
**Ce qu'il montre :** chaque barre est un livre, chaque segment la contribution d'une composante (poids × similarité calibrée), la pénalité « à éviter » part vers la gauche (en rouge), le losange noir est le score final.
**Pourquoi c'est essentiel :** le sujet demande de « démontrer comment les résultats découlent de l'analyse sémantique ». La somme des segments retombe sur le score final : la recommandation est **traçable** de bout en bout. On voit aussi immédiatement quel critère a fait gagner un livre.

In [ ]:
fig_breakdown = viz.score_breakdown(recs)
fig_breakdown.show()

# Vérification : la somme des contributions = le score final (au plafonnement [0, 1] près)
for r in recs:
    print(f"{r.title[:40]:40s}  somme des contributions = {sum(r.contributions.values()):.3f}   score final = {r.final_score:.3f}")

### 3.4 Carte sémantique du catalogue — `similarity_map(coords, genres, titres, ...)`
**Ce qu'elle montre :** les 12 000 livres projetés en 2D à partir de leurs embeddings SBERT (UMAP, métrique cosinus ; repli PCA si `umap-learn` est absent), colorés par genre principal. L'utilisateur (✖) est placé au barycentre pondéré de ses 15 livres les plus proches, le top 3 est marqué par des étoiles, les plus proches voisins par des cercles.
**Interprétation :** deux livres proches racontent des histoires de sens proche, même sans mots communs. Les genres forment des régions cohérentes, ce qui valide visuellement les embeddings ; l'utilisateur tombe dans la région qui correspond à sa demande.
**Choix technique :** placer l'utilisateur par `umap.transform` demanderait de conserver le modèle UMAP entraîné (lourd) et donnerait une position instable ; le barycentre des plus proches voisins est simple, déterministe et fidèle à l'idée « vous êtes là où sont les livres qui vous ressemblent ».
Les coordonnées sont calculées une seule fois puis mises en cache dans `data/embeddings/umap_2d.npy` (+ `umap_2d_book_ids.json` pour garantir l'alignement).

In [ ]:
coords = viz.load_or_compute_map_coords(matrix, books["book_id"].astype(str).tolist())
q = profile.query_vecs.get("global")
if q is None:                                   # au cas où P4 ne renverrait pas le vecteur
    q = engine.encode([profile.query_texts["global"]])[0]
sims = matrix @ np.asarray(q).ravel()
user_xy, nn = viz.user_map_position(coords, sims, k=15)

idx = {b: i for i, b in enumerate(books["book_id"].astype(str))}
fig_map = viz.similarity_map(
    coords, [viz.primary_genre(g) for g in books["genres"]], books["title"].tolist(),
    highlight=[idx[r.book_id] for r in recs if r.book_id in idx], highlight_labels=[r.title for r in recs],
    user_xy=user_xy, neighbours=nn,
)
fig_map.show()

### 3.5 Heatmap de similarité — `similarity_heatmap(sim, lignes, colonnes)`
**Ce qu'elle montre :** le cosinus entre les **mots de l'utilisateur** (lignes : réponses Q1-Q3 et phrases de préférence par bloc) et **ce que l'enrichissement NLP a attribué au livre** (colonnes : ses labels de genre, thèmes, style, atmosphère, représentés par les mêmes vecteurs que le moteur, plus la phrase du résumé la plus proche).
**Pourquoi c'est la preuve de l'analyse sémantique :** une case foncée relie deux formulations différentes d'une même idée. C'est exactement ce que le sujet appelle « not purely keyword matching ».

In [ ]:
book = books.iloc[idx[recs[0].book_id]]
ft = answers["free_text"]
rows = [(f"Q{i}: {ft[k]}", ft[k]) for i, k in ((1, "ideal_book"), (2, "loved_book"), (3, "atmosphere")) if ft[k]]
# phrases de préférence par bloc (P4 : profile.block_phrases ; 3 max par bloc pour rester lisible)
phrases = getattr(profile, "block_phrases", None) or {}
for b in config.BLOCKS:
    items = phrases.get(b) or [t.strip() for t in (profile.query_texts.get(b) or "").replace(" | ", ";").split(";") if t.strip()]
    rows += [(f"{b}: {t}", t) for t in items[:3]]

cols, vecs = [], []
for b, col in (("Genre", "genres"), ("Themes", "themes"), ("Style", "styles"), ("Atmosphere", "atmospheres")):
    names, mat = engine.label_matrix(b)
    for label in book[col]:
        cols.append(f"{b} · {label}")
        vecs.append(mat[names.index(label)])
cols.append("Closest sentence")
vecs.append(engine.encode([recs[0].best_sentence])[0])

sim = engine.encode([t for _, t in rows]) @ np.vstack(vecs).T
fig_heat = viz.similarity_heatmap(sim, [r for r, _ in rows], cols, title=f"Your words × {recs[0].title}")
fig_heat.show()

## 4. Explicabilité dans les cartes de recommandation

Chaque carte du top 3 affiche les **4 éléments exigés** (item, score, critères principaux, explication) :
- **l'item** : couverture, titre, auteurs ;
- **le score** : jauge du score final + badge *Excellent / Good / Partial match* (seuils 0,75 / 0,55 de `config.MATCH_THRESHOLDS`) + score de couverture ;
- **les critères principaux** : puces colorées des labels communs avec leur similarité (`matched_labels`) ;
- **l'explication** générée par template (`explanation`, P4), précédée de la **phrase du résumé la plus proche** de la demande (`best_sentence`), surlignée aussi dans le résumé complet ;
- un panneau dépliable **« How was this score computed? »** : la formule (LaTeX) et un tableau score / poids / contribution pour chaque composante.

In [ ]:
for r in recs:
    print(f"#{r.rank} {r.title} — {r.authors}  [{r.match_label}, {r.final_score:.0%}]")
    print("   Phrase la plus proche :", r.best_sentence[:150])
    print("   Critères :", {b: [l for l, _ in v] for b, v in r.matched_labels.items()})
    print("   Explication :", r.explanation[:300], "\n")

## 5. Export des figures pour le rapport et les slides

`viz.save_figure` écrit des PNG haute résolution dans `docs/figures/` (préfixe `05_`) via **kaleido**.
Avec kaleido ≥ 1.0, un Chrome est nécessaire : si l'export échoue, lancer `plotly_get_chrome` une fois.

In [ ]:
paths = viz.export_all({
    "05_radar_profile": fig_profile,
    "05_radar_compare": fig_compare,
    "05_score_breakdown": fig_breakdown,
    "05_similarity_map": fig_map,
    "05_similarity_heatmap": fig_heat,
})
paths

## 6. Captures d'écran de l'application

Captures générées automatiquement par `python -m scripts.capture_screens` (l'app doit tourner dans un autre terminal ; pré-requis : `pip install playwright && python -m playwright install chromium`).

| Écran | Ce qu'il montre |
|---|---|
| ![](../docs/figures/05_screen_1_free_text.png) | **Étape 1** : 4 questions ouvertes, Q1 obligatoire avec compteur de caractères, exemples en placeholder. Barre latérale : progression, navigation, mode démo. |
| ![](../docs/figures/05_screen_2_genres_themes.png) | **Étape 2** : cases à cocher genres et thèmes ; la bulle ⓘ affiche la phrase descriptive réellement encodée par le moteur. |
| ![](../docs/figures/05_screen_3_context.png) | **Étape 3** : époque du récit (radio), public (selectbox), formats (multiselect). |
| ![](../docs/figures/05_screen_4_likert.png) | **Étape 4** : échelles de Likert avec libellés aux deux extrémités ; importance des 4 blocs (poids W_b). |
| ![](../docs/figures/05_screen_5_guided.png) | **Étape 5** : questions guidées ; l'auteur favori est recherché en direct dans le catalogue. |
| ![](../docs/figures/05_screen_6_top3.png) | **Résultats, top 3** : carte par livre (score, badge, phrase la plus proche, explication, critères), radar et décomposition du score. |
| ![](../docs/figures/05_screen_7_your_profile.png) | **Profil lecteur** : résumé, radars par bloc, poids, filtres appliqués. |
| ![](../docs/figures/05_screen_7_compare.png) | **Comparaison** des 3 livres : barres empilées + radar commun. |
| ![](../docs/figures/05_screen_7_semantic_map.png) | **Carte sémantique** du catalogue avec l'utilisateur et le top 3. |
| ![](../docs/figures/05_screen_7_under_the_hood.png) | **Sous le capot** : heatmap mots de l'utilisateur × labels du livre. |

## 7. Robustesse et choix d'ergonomie
- **Chargement unique** des ressources lourdes (`st.cache_resource`) : modèle SBERT, catalogue, matrice des livres, carte 2D. Spinner explicite au premier lancement ; `scripts/prepare_app.py` pré-calcule tout avant la démo.
- **Aucun résultat après filtres** (public + format + auteur exclu trop restrictifs) : message clair et bouton « Relax the filters ».
- **Moteur indisponible** (pas d'Internet au premier lancement, erreur de P4) : l'app ne plante pas, elle bascule sur le stub ou désactive la carte avec un message.
- **Mode démo** : 3 personas pré-remplis (barre latérale, ou lien direct `?persona=Young%20fantasy%20fan`) pour une démo fiable en soutenance.
- **Persistance** : les réponses survivent à la navigation entre étapes, peuvent être rechargées depuis un `answers.json`, et les résultats sont téléchargeables en JSON (utile pour l'évaluation de P6).